# 教学模块二：构建EEG分类模型 (EEGBiFormer)
# Teaching Module 2: Building the EEG Classification Model (EEGBiFormer)

欢迎来到项目的第二部分！

Welcome to the second part of the project!

在数据预处理模块中，我们成功地将原始、嘈杂的EEG信号转换成了干净、格式统一的数据样本（Tensor）。

In the data preprocessing module, we successfully transformed raw, noisy EEG signals into clean, uniformly formatted data samples (Tensors).

现在，我们的任务是构建一个足够聪明的“大脑”——也就是深度学习模型——来学习这些数据中的深层模式，并最终区分出“专注”与“放松”这两种状态。

Now, our task is to build a sufficiently intelligent "brain"—a deep learning model—to learn the deep patterns within this data and ultimately distinguish between the "focus" and "rest" states.

### 本模块学习目标：
### Learning Objectives for This Module:

1.  **理解混合模型架构**：为什么我们将卷积神经网络（CNN）和Transformer这两种强大的技术结合起来？

1  Understand the Hybrid Model Architecture**: Why do we combine two powerful technologies like Convolutional Neural Networks (CNNs) and Transformers?

2.  **掌握CNN的角色**：学习一维CNN（`Conv1d`）如何从EEG时间序列中提取局部特征。

2  **Master the Role of CNN**: Learn how a 1D CNN (`Conv1d`) extracts local features from EEG time-series data.

3.  **掌握Transformer的角色**：学习Transformer的自注意力机制如何捕捉特征之间的长期依赖关系。

3  **Master the Role of the Transformer**: Learn how the Transformer's self-attention mechanism captures long-term dependencies between features.

4.  **PyTorch模型实现**：亲手编写一个完整的、继承自 `nn.Module` 的PyTorch模型。

4  **Implement a PyTorch Model**: Write a complete PyTorch model from scratch that inherits from `nn.Module`.

5.  **模型验证与检查**：学习如何实例化模型并检查其结构，确保它能正确处理我们的数据。

5  **Validate and Inspect the Model**: Learn how to instantiate a model and check its structure to ensure it can correctly process our data.

## 原理剖析（一）：CNN —— 局部的特征提取大师

## Principle Analysis (Part 1): CNN — The Master of Local Feature Extraction

首先，模型的第一部分是**一维卷积神经网络 (1D-CNN)**。

First, the initial part of the model is a **1D Convolutional Neural Network (1D-CNN)**.

#### 为什么用CNN？

#### Why use a CNN?

想象一下，我们用一个很小的“特征检测窗口”（称为**卷积核**或**滤波器**）沿着EEG信号的时间轴滑动。

Imagine sliding a small "feature detection window" (called a **kernel** or **filter**) along the time axis of the EEG signal.

在每个位置，这个窗口都会计算出一个值，代表它与当前信号片段的匹配程度。

At each position, this window calculates a value representing how well it matches the current signal segment.

通过训练，这些“特征检测窗口”会自动学会识别EEG信号中一些有意义的**局部基本模式**，例如：

Through training, these "feature detection windows" automatically learn to recognize meaningful **local, fundamental patterns** in the EEG signal, such as:

- 特定频率的振荡波形（如Alpha波、Beta波的片段）

- Oscillatory waveforms of specific frequencies (like segments of Alpha or Beta waves).

- 信号中的尖峰或波谷


- Spikes or troughs in the signal.

- 波形的特定变化趋势

- Specific trends in the waveform's shape.

CNN的优势在于它的**平移不变性**（无论一个特征出现在信号的第0.1秒还是第1.5秒，都能被识别出来）和**层级特征提取**能力（浅层网络学习简单波形，深层网络将它们组合成更复杂的模式）。

The advantages of CNNs lie in their **translation invariance** (a feature can be recognized whether it appears at 0.1 seconds or 1.5 seconds into the signal) and their ability for **hierarchical feature extraction** (shallow layers learn simple waveforms, while deeper layers combine them into more complex patterns).

在我们的模型中，我们堆叠了三个`Conv1d`层，逐步从16个原始通道中提取出越来越抽象和复杂的特征，最终将信号编码成一个256维的特征序列。

In our model, we stack three `Conv1d` layers to progressively extract more abstract and complex features from the initial 16 channels, ultimately encoding the signal into a 256-dimensional feature sequence.

同时，我们使用**批标准化 (`BatchNorm1d`)** 和 **ReLU激活函数** 来稳定和加速训练过程。

Meanwhile, we use **Batch Normalization (`BatchNorm1d`)** and the **ReLU activation function** to stabilize and accelerate the training process.

## 原理剖析（二）：Transformer —— 全局的关联建模专家

## Principle Analysis (Part 2): Transformer — The Expert in Global Relationship Modeling

经过CNN处理后，我们得到了一系列特征向量，每个向量代表一个时间步上的局部信息。

After processing with the CNN, we obtain a series of feature vectors, each representing local information at a specific time step.

但是，一个认知状态（比如“专注”）是持续的，不同时间点的信息并非孤立的。

However, a cognitive state (like "focus") is continuous, and the information at different time points is not isolated.

这时，**Transformer编码器**就登场了。

This is where the **Transformer Encoder** comes in.

它的核心是**自注意力机制 (Self-Attention)**。

Its core is the **self-attention mechanism**.

#### 什么是自注意力？

#### What is Self-Attention?

自注意力机制允许模型在处理一个时间点的特征时，同时“关注”到序列中所有其他时间点的特征，并根据相关性为它们分配不同的“注意力权重”。

The self-attention mechanism allows the model, when processing a feature at one time point, to simultaneously "attend" to features at all other time points in the sequence and assign different "attention weights" based on their relevance.

**举个例子**：假设在2秒的信号片段中，第0.5秒和第1.8秒出现的脑电模式对于判断“专注”状态至关重要。

**For example**: Assume that in a 2-second signal segment, the brainwave patterns appearing at 0.5 seconds and 1.8 seconds are crucial for determining the "focus" state.

自注意力机制就能学会，在做最终决策时，给予这两个时间点提取出的特征更高的权重，而忽略那些不太相关的片段。

The self-attention mechanism can learn to assign higher weights to the features extracted at these two time points when making a final decision, while ignoring less relevant segments.

#### CNN + Transformer = 强强联合

#### CNN + Transformer = A Powerful Combination

- **CNN** 负责“微观”：从原始信号中高效地提取出干净、有意义的局部特征。

- **CNN** handles the "micro" level: efficiently extracting clean and meaningful local features from the raw signal.

- **Transformer** 负责“宏观”：分析这些局部特征组成的序列，并建立它们之间的长期依赖关系，理解整个时间窗口内的“上下文”。

- **Transformer** handles the "macro" level: analyzing the sequence of these local features and establishing long-term dependencies between them to understand the "context" of the entire time window.

这种组合拳使得模型既能捕捉到信号的精细波形特征，又能理解这些特征在时间维度上的复杂关联，非常适合EEG这类时序信号的分析。

This combined approach enables the model to capture both the fine-grained waveform features of the signal and understand their complex relationships over time, making it highly suitable for analyzing time-series signals like EEG.

#### 分类头：最后通过一个全连接网络将Transformer的输出映射到最终的类别预测上。

#### Classification Head: Finally, a fully connected network maps the Transformer's output to the final class predictions.

#### 最终数据流

#### Final Data Flow

`输入信号` -> `批标准化` -> `CNN特征提取器` -> `Transformer编码器` -> `全局平均池化` -> `分类头` -> `输出类别`

`Input Signal` -> `Batch Norm` -> `CNN Feature Extractor` -> `Transformer Encoder` -> `Global Average Pooling` -> `Classification Head` -> `Output Classes`

In [1]:
# 导入 PyTorch 核心库
# Import core PyTorch libraries
import torch
# 导入 PyTorch 的神经网络模块，我们用它来构建模型层
# Import PyTorch's neural network module, which we use to build model layers
import torch.nn as nn

# 定义 EEGBiFormer 模型类，它继承自 PyTorch 的 nn.Module
# Define the EEGBiFormer model class, which inherits from nn.Module
class EEGBiFormer(nn.Module):
    """
    自定义的EEG分类模型，结合了CNN和Transformer。
    A custom EEG classification model that combines CNN and Transformer.
    """
    # 定义构造函数，初始化模型的各个层
    # Define the constructor to initialize the model's layers
    def __init__(self, num_classes, in_channels=16, dim=256):
        # 调用父类 nn.Module 的构造函数，这是自定义模块所必需的
        # Call the constructor of the parent class, nn.Module, which is necessary for custom modules
        super().__init__()
        
        # --- 输入数据的批标准化层 ---
        # --- Input Data Batch Normalization Layer ---
        # 对输入的16个通道进行标准化，增强训练稳定性
        # Normalize the 16 input channels to enhance training stability
        self.input_norm = nn.BatchNorm1d(in_channels)
        
        # --- CNN 特征提取器模块 ---
        # --- CNN Feature Extractor Module ---
        # 使用 nn.Sequential 将多个层组合成一个模块
        # Use nn.Sequential to group multiple layers into a single module
        self.feature_extractor = nn.Sequential(
            # 第一个卷积层：输入通道16，输出通道64，卷积核大小9，padding确保输出长度不变，不使用偏置项
            # First convolutional layer: 16 input channels, 64 output channels, kernel size of 9, padding ensures output length is unchanged, no bias term
            nn.Conv1d(in_channels, 64, kernel_size=9, padding=4, bias=False),
            # 对64个通道进行批标准化
            # Batch normalization for the 64 channels
            nn.BatchNorm1d(64),
            # ReLU 激活函数，引入非线性
            # ReLU activation function to introduce non-linearity
            nn.ReLU(),
            # 第二个卷积层：输入通道64，输出通道128
            # Second convolutional layer: 64 input channels, 128 output channels
            nn.Conv1d(64, 128, kernel_size=9, padding=4, bias=False),
            # 对128个通道进行批标准化
            # Batch normalization for the 128 channels
            nn.BatchNorm1d(128),
            # ReLU 激活函数
            # ReLU activation function
            nn.ReLU(),
            # 第三个卷积层：输入通道128，输出通道dim (256)
            # Third convolutional layer: 128 input channels, dim (256) output channels
            nn.Conv1d(128, dim, kernel_size=9, padding=4, bias=False),
            # 对dim(256)个通道进行批标准化
            # Batch normalization for the dim(256) channels
            nn.BatchNorm1d(dim),
            # ReLU 激活函数
            # ReLU activation function
            nn.ReLU()
        )
        
        # --- Transformer 编码器模块 ---
        # --- Transformer Encoder Module ---
        # 定义一个标准的 Transformer 编码器层
        # Define a standard Transformer Encoder Layer
        encoder_layer = nn.TransformerEncoderLayer(
            # 输入特征的维度 (必须与CNN输出维度一致)
            # The dimension of input features (must match the CNN output dimension)
            d_model=dim,
            # 多头注意力机制中的头数
            # The number of heads in the multi-head attention mechanism
            nhead=8,
            # 前馈网络层的隐藏层维度，通常设为 d_model 的4倍
            # The dimension of the hidden layer in the feed-forward network, typically set to 4 times d_model
            dim_feedforward=dim*4,
            # Dropout 比例，防止过拟合
            # Dropout rate to prevent overfitting
            dropout=0.2,
            # 关键参数！确保输入张量的第一个维度是 batch_size
            # Crucial parameter! Ensures the first dimension of the input tensor is the batch_size
            batch_first=True
        )
        # 将多个编码器层堆叠起来（这里我们使用2层）
        # Stack multiple encoder layers together (here we use 2 layers)
        self.transformer = nn.TransformerEncoder(encoder_layer, num_layers=2)
        
        # --- 分类头模块 ---
        # --- Classification Head Module ---
        # 用于将 Transformer 的输出映射到最终的类别得分
        # Used to map the Transformer's output to the final class scores
        self.head = nn.Sequential(
            # 全连接层，从 dim 维降到 dim/2 维
            # Fully connected layer, reducing dimension from dim to dim/2
            nn.Linear(dim, dim//2),
            # ReLU 激活函数
            # ReLU activation function
            nn.ReLU(),
            # Dropout 层，进一步防止过拟合
            # Dropout layer to further prevent overfitting
            nn.Dropout(0.2),
            # 输出层，维度为类别数
            # Output layer, with dimension equal to the number of classes
            nn.Linear(dim//2, num_classes)
        )
        
        # 应用自定义的权重初始化方法，有助于稳定训练
        # Apply a custom weight initialization method to help stabilize training
        self.apply(self._initialize_weights)
        # 打印一条消息，确认模型创建成功
        # Print a message to confirm successful model creation
        print("The EEGBiFormer model is created and the weights are initialized using a custom method。")

    # 定义一个私有方法用于权重初始化
    # Define a private method for weight initialization
    def _initialize_weights(self, m):
        # 这是一个自定义的权重初始化函数
        # This is a custom weight initialization function
        if isinstance(m, nn.Conv1d): # 检查模块是否是 Conv1d 类型
            # Check if the module is of type Conv1d
            # 对卷积层使用 Kaiming He 初始化，非常适合ReLU激活函数
            # Use Kaiming He initialization for convolutional layers, which is well-suited for ReLU activation functions
            nn.init.kaiming_normal_(m.weight, mode='fan_out', nonlinearity='relu')
        elif isinstance(m, nn.BatchNorm1d): # 检查模块是否是 BatchNorm1d 类型
            # Check if the module is of type BatchNorm1d
            # 对批标准化层的权重初始化为1，偏置初始化为0
            # Initialize the weights of batch normalization layers to 1 and biases to 0
            nn.init.constant_(m.weight, 1)
            nn.init.constant_(m.bias, 0)
        elif isinstance(m, nn.Linear): # 检查模块是否是 Linear 类型
            # Check if the module is of type Linear
            # 对全连接层的权重使用小的正态分布进行初始化
            # Initialize the weights of fully connected layers using a small normal distribution
            nn.init.normal_(m.weight, 0, 0.01)
            # 如果存在偏置，则初始化为0
            # If bias exists, initialize it to 0
            if m.bias is not None: nn.init.constant_(m.bias, 0)

    # 定义模型的前向传播逻辑
    # Define the model's forward pass logic
    def forward(self, x):
        # --- 定义模型的前向传播过程 ---
        # --- Define the model's forward propagation process ---
        # x 的输入形状: (Batch, Time, Channels) -> (N, 404, 16)
        # Input shape of x: (Batch, Time, Channels) -> (N, 404, 16)
        
        # --- 预处理与维度调整 ---
        # --- Preprocessing and Dimension Adjustment ---
        # PyTorch 的卷积层和BN层期望通道数在第二个维度
        # PyTorch's convolutional and batch normalization layers expect the number of channels to be the second dimension
        # 因此需要将 x 的形状从 (N, L, C) 转置为 (N, C, L)
        # Therefore, the shape of x needs to be transposed from (N, L, C) to (N, C, L)
        x = x.transpose(1, 2) # -> (N, 16, 404)
        
        # 应用输入批标准化
        # Apply input batch normalization
        x = self.input_norm(x)
        
        # --- CNN 特征提取 ---
        # --- CNN Feature Extraction ---
        # 通过 CNN 特征提取器
        # Pass through the CNN feature extractor
        x = self.feature_extractor(x) # -> (N, 256, 404)
        
        # --- Transformer 处理 ---
        # --- Transformer Processing ---
        # Transformer 层期望序列长度在第二个维度
        # The Transformer layer expects the sequence length to be the second dimension
        # 因此需要将 x 的形状从 (N, C, L) 再次转置为 (N, L, C)
        # Therefore, the shape of x needs to be transposed again from (N, C, L) to (N, L, C)
        x = x.transpose(1, 2) # -> (N, 404, 256)
        
        # 通过 Transformer 编码器
        # Pass through the Transformer encoder
        x = self.transformer(x) # -> (N, 404, 256)
        
        # --- 分类 ---
        # --- Classification ---
        # 对序列维度（时间维度）进行平均池化，得到一个固定大小的表示向量
        # Perform average pooling over the sequence dimension (time dimension) to get a fixed-size representation vector
        # 形状从 (N, L, C) 变为 (N, C)
        # The shape changes from (N, L, C) to (N, C)
        x = x.mean(dim=1) # -> (N, 256)
        
        # 通过分类头得到最终的类别得分
        # Get the final class scores through the classification head
        return self.head(x) # -> (N, num_classes)

## 步骤四：实例化并检查模型

## Step 4: Instantiating and Inspecting the Model

现在我们已经定义好了 `EEGBiFormer` 类，让我们来实际创建一个模型实例，并检查一下它的内部结构。

Now that we have defined the `EEGBiFormer` class, let's actually create a model instance and inspect its internal structure.

我们还会创建一个虚拟的输入数据（dummy input），将它送入模型，检查输出的形状是否符合我们的预期。

We will also create a dummy input, feed it into the model, and check if the output shape meets our expectations.

这是一种非常好的调试和验证模型的习惯。

This is a very good habit for debugging and validating a model.

In [2]:
# --- 设置设备 ---
# --- Set Device ---
# 检查是否有可用的 CUDA (GPU)，如果有则使用'cuda'，否则使用'cpu'
# Check for an available CUDA-enabled GPU; if available, use 'cuda', otherwise use 'cpu'
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
# 打印将要使用的设备信息
# Print the device that will be used
print(f"device will be used: {device}\\n")

# --- 实例化模型 ---
# --- Instantiate the Model ---
# 假设我们的任务是二分类（focus vs rest），所以 num_classes=2
# Assuming our task is binary classification (focus vs rest), so num_classes=2
num_classes = 2
# 创建 EEGBiFormer 模型实例，并将其移动到指定的设备（GPU或CPU）
# Create an instance of the EEGBiFormer model and move it to the specified device (GPU or CPU)
model = EEGBiFormer(num_classes=num_classes).to(device)

# --- 打印模型结构 ---
# --- Print Model Structure ---
# 这会清晰地展示出模型的所有层和参数
# This will clearly display all the layers and parameters of the model
print("--- Model structure ---")
# 打印模型对象，PyTorch 会自动格式化其结构
# Print the model object; PyTorch will automatically format its structure
print(model)

# --- 创建一个虚拟输入张量进行测试 ---
# --- Create a Dummy Input Tensor for Testing ---
# 形状需要和我们的数据样本一致：(batch_size, window_size, in_channels)
# The shape needs to be consistent with our data samples: (batch_size, window_size, in_channels)
# 定义批次大小为4
# Define the batch size as 4
batch_size = 4
# 定义窗口大小（时间点数）为404
# Define the window size (number of time points) as 404
window_size = 404
# 定义输入通道数为16
# Define the number of input channels as 16
in_channels = 16
# 创建一个符合形状要求的、填充了随机数的张量作为虚拟输入数据
# Create a randomly filled tensor with the required shape as dummy input data
# 并将其移动到指定设备
# and move it to the specified device
dummy_input = torch.randn(batch_size, window_size, in_channels).to(device)

# 打印一个标题，表示开始进行前向传播测试
# Print a title to indicate the start of the forward pass test
print(f"\n--- Forward propagation test ---")
# 打印虚拟输入张量的形状，以确认其符合预期
# Print the shape of the dummy input tensor to confirm it meets expectations
print(f"The shape of the input tensor: {dummy_input.shape}")

# 将模型设置为评估模式 (这会关闭 Dropout 等只在训练时使用的层)
# Set the model to evaluation mode (this disables layers like Dropout that are only used during training)
model.eval()
# 使用 torch.no_grad() 上下文管理器，在该块内的计算不会被跟踪梯度，可以节省内存和计算
# Use the torch.no_grad() context manager; computations within this block will not track gradients, saving memory and computation
with torch.no_grad():
    # 将虚拟数据送入模型，执行一次前向传播
    # Feed the dummy data into the model to perform one forward pass
    output = model(dummy_input)

# 打印模型输出张量的形状
# Print the shape of the model's output tensor
print(f"The shape of the output tensor: {output.shape}")

# --- 验证输出 ---
# --- Validate Output ---
# 对于一个batch_size=4的二分类任务，我们期望的输出形状是 (4, 2)
# For a binary classification task with batch_size=4, the expected output shape is (4, 2)
if output.shape == (batch_size, num_classes): # 检查输出形状是否与预期完全一致
    # Check if the output shape is exactly as expected
    # 如果一致，打印测试通过的消息
    # If it matches, print a success message
    print("\nThe test passed! The input and output shapes of the model meet expectations.")
else:
    # 如果不一致，打印测试失败的消息，并显示期望和实际的形状
    # If it doesn't match, print a failure message showing the expected and actual shapes
    print(f"\nTest failed! Expected output shape (batch_size, num_classes)={(batch_size, num_classes)}，But actually get {output.shape}")

device will be used: cuda\n
The EEGBiFormer model is created and the weights are initialized using a custom method。
--- Model structure ---
EEGBiFormer(
  (input_norm): BatchNorm1d(16, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
  (feature_extractor): Sequential(
    (0): Conv1d(16, 64, kernel_size=(9,), stride=(1,), padding=(4,), bias=False)
    (1): BatchNorm1d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (2): ReLU()
    (3): Conv1d(64, 128, kernel_size=(9,), stride=(1,), padding=(4,), bias=False)
    (4): BatchNorm1d(128, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (5): ReLU()
    (6): Conv1d(128, 256, kernel_size=(9,), stride=(1,), padding=(4,), bias=False)
    (7): BatchNorm1d(256, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (8): ReLU()
  )
  (transformer): TransformerEncoder(
    (layers): ModuleList(
      (0): TransformerEncoderLayer(
        (self_attn): MultiheadAttention(
         

## 模块二总结

## Module 2 Summary

我们构建并验证了一个复杂的混合深度学习模型。

We built and validated a complex hybrid deep learning model.

核心回顾：

Core Review:

- **混合架构**：我们结合了CNN的局部特征提取能力和Transformer的全局上下文建模能力。

- **Hybrid Architecture**: We combined the local feature extraction capabilities of CNNs with the global context modeling power of Transformers.

- **PyTorch实现**：我们通过定义 `__init__` (构建层) 和 `forward` (定义数据流) 方法，创建了一个自定义的 `nn.Module`。

- **PyTorch Implementation**: We created a custom `nn.Module` by defining the `__init__` (to build layers) and `forward` (to define data flow) methods.

- **维度转换**：我们特别注意了在使用CNN和Transformer时，需要通过 `transpose` 操作来调整数据张量的维度顺序。

- **Dimension Transformation**: We paid special attention to the need to use the `transpose` operation to adjust the dimensional order of the data tensor.

- **模型检查**：我们学会了如何打印模型结构和使用数据进行前向传播测试，这是确保模型正确性的重要步骤。

- **Model Inspection**: We learned how to print the model structure and use data for a forward pass test, which is a crucial step in ensuring the model's correctness.

现在，我们既有了高质量的数据（模块一），也有了强大的模型（模块二）。

Now, we have both high-quality data (from Module 1) and a powerful model (from Module 2).

在下一个，也是最后一个核心模块中，我们将把这两者结合起来，编写训练循环，真正地开始**训练模型**！

In the next and final core module, we will combine these two, write the training loop, and truly begin to **train the model**!